[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ELTE-DSED/Intro-Data-Security/blob/main/modules/module_05_confidentiality/Lab_5a_Membership_Inference_Attacks.ipynb)

# **Lab 5a: Membership Inference Attacks with ART**

**Master's Course:** Introduction to Data Security Practicum  
**Institution:** Eötvös Loránd University (ELTE), Faculty of Informatics (IK)  
**Module 5:** Confidentiality & Privacy Auditing


## **Learning Objectives**

By the end of this lab, you will be able to:

- **Analyze Threat Models:** Formulate membership inference threat vectors across black-box probability query APIs and hard-label prediction services.
- **Quantify Generalization Vulnerability:** Demonstrate how generalization error (overfitting) creates identifiable statistical confidence gaps between training members and unseen non-members.
- **Implement Manual MIA:** Build and evaluate confidence thresholding and loss-based decision rules to compute ROC-AUC curves and Membership Advantage.
- **Utilize ART Inference Framework:** Execute standardized black-box rule-based membership inference attacks using the Adversarial Robustness Toolbox.
- **Execute Label-Only Attacks:** Probe decision boundaries via iterative input perturbations to infer membership when confidence scores and probabilities are withheld.
- **Evaluate Privacy Defenses:** Contrast the privacy-preserving efficacy of classical regularization (L2 weight decay, dropout) against formal differential privacy guarantees.


## **Table of Contents**

1. [Threat Model: Privacy Leakage](#threat-model)
2. [Membership Inference Theory & Generalization Gap](#theory)
3. [Setup & Data Preparation](#setup)
4. [Model Architectures & Training Regimes](#training)
5. [Manual Confidence-Based Membership Inference](#confidence)
6. [ART Membership Inference Framework](#art)
7. [Label-Only Decision Boundary Attacks](#label-only)
8. [Hands-on Exploratory Exercises](#exercises)
9. [Conclusion & Next Steps](#conclusion)


## **1. Threat Model: Privacy Leakage** <a name="threat-model"></a>

A **Membership Inference Attack (MIA)** asks a fundamental and privacy-critical question:

> *"Was sample $\mathbf{x}$ part of the training set $\mathcal{D}_{train}$?"*

The attack was formally introduced by [Shokri et al. (2017)](https://doi.org/10.48550/arXiv.1610.05820) and works by exploiting the **generalization gap**: machine learning models tend to exhibit higher prediction confidence and lower loss on training records than on unseen test samples.

### **Real-World Impact**

| Domain | Sensitive Context | Privacy Risk | Potential Consequence |
|:---|:---|:---|:---|
| **Healthcare** | Disease classification model | Attacker determines patient $P$ was in clinical trial | Exposes patient's positive diagnosis |
| **Finance** | Fraud or credit scoring | Attacker determines user transaction was used for training | Exposes credit status or account holder identity |
| **Facial Recognition** | Biometric surveillance database | Attacker checks whether person $A$ is registered | Confirms presence in sensitive monitored cohort |
| **Genomics** | Genome-wide association study (GWAS) | Attacker checks if individual's DNA trained marker model | Reveals genetic predisposition to conditions |

### **Threat Assumptions**

- **Black-Box Access:** The adversary submits queries $\mathbf{x}$ to the deployed model and observes outputs (class probabilities, confidence vectors, or hard predicted labels).
- **Target Knowledge:** The adversary possesses the candidate sample $\mathbf{x}$ and its ground truth or suspected class label $y$.
- **Auxiliary Knowledge:** In advanced setups (shadow models or calibrated label-only attacks), the adversary has access to a small auxiliary dataset from the same underlying distribution to calibrate decision thresholds.


## **2. Membership Inference Theory & Generalization Gap** <a name="theory"></a>

### **Why Does Membership Inference Work?**

Empirical risk minimization pushes the training loss toward zero. When a model overfits, it effectively memorizes idiosyncratic quirks of the training records. Consequently:

$$\mathbb{E}_{(\mathbf{x}, y) \sim \mathcal{D}_{train}}[f_y(\mathbf{x})] > \mathbb{E}_{(\mathbf{x}, y) \sim \mathcal{D}_{test}}[f_y(\mathbf{x})]$$

where $f_y(\mathbf{x})$ denotes the model's predicted confidence for the true class $y$.

<img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/modules/module_05_confidentiality/images/bias_variance_tradeoff.png" width="75%" style="display:block; margin:auto;">

### **Attack Principle: Members vs. Non-Members**

<img src="https://raw.githubusercontent.com/ELTE-DSED/Intro-Data-Security/main/modules/module_05_confidentiality/images/attack_principle_member_vs_nonmember.png" width="75%" style="display:block; margin:auto;">

- **Training Member:** Model output distribution is sharply peaked at the ground-truth class (high confidence, low entropy, low cross-entropy loss).
- **Unseen Non-Member:** Model output distribution exhibits higher uncertainty (lower confidence, higher entropy, higher cross-entropy loss).


## **3. Setup & Data Preparation** <a name="setup"></a>

We configure our environment, install required libraries, and prepare a tabular clinical dataset: the **Breast Cancer Wisconsin** dataset ($n=569$, $d=30$ diagnostic features).

We intentionally partition the dataset so that our training set is relatively small, accentuating the contrast between overfitted and regularized models.


In [ ]:
# Install required dependencies for Google Colab
%pip install -q adversarial-robustness-toolbox scikit-learn matplotlib tqdm


In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from typing import Tuple, List
from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

np.random.seed(42)
torch.manual_seed(42)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Device: {device}")

In [ ]:
# Load Breast Cancer Wisconsin dataset
data = load_breast_cancer()
X = data.data.astype(np.float32)
y = data.target.astype(np.int64)

# Split into candidate members and held-out non-members
X_train_full, X_temp, y_train_full, y_temp = train_test_split(
    X, y, test_size=0.4, random_state=42, stratify=y
)
X_val, X_test, y_val, y_test = train_test_split(
    X_temp, y_temp, test_size=0.5, random_state=42, stratify=y_temp
)

# Use a very small member set to encourage memorization and stronger leakage
member_train_size = 40
X_train, _, y_train, _ = train_test_split(
    X_train_full, y_train_full, train_size=member_train_size, random_state=42, stratify=y_train_full
)

# Standardize features using member-train statistics only
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train).astype(np.float32)
X_val = scaler.transform(X_val).astype(np.float32)
X_test = scaler.transform(X_test).astype(np.float32)

train_data = TensorDataset(torch.from_numpy(X_train), torch.from_numpy(y_train))
val_data = TensorDataset(torch.from_numpy(X_val), torch.from_numpy(y_val))
test_data = TensorDataset(torch.from_numpy(X_test), torch.from_numpy(y_test))

train_loader = DataLoader(train_data, batch_size=32, shuffle=True)
val_loader = DataLoader(val_data, batch_size=64, shuffle=False)
test_loader = DataLoader(test_data, batch_size=64, shuffle=False)

input_dim = X_train.shape[1]
print(
    f"Training set (members): {len(train_data)}, Validation set: {len(val_data)}, "
    f"\nTest set (non-members): {len(test_data)}, Features: {input_dim}"
    f"\nClasses: {data.target_names.tolist()}"
)

## **4. Model Architectures & Training Regimes** <a name="training"></a>

To demonstrate how generalization error dictates privacy leakage, we construct two identical classification tasks under two contrasting training regimes:

1. **Overfitted Model (`overfit_model`):** A large 3-layer MLP trained for 200 epochs without weight decay or dropout. It memorizes the training records, yielding near 100% training accuracy but a noticeable test generalization gap.
2. **Regularized Model (`reg_model`):** A compact MLP trained with explicit $L_2$ weight decay ($\lambda = 0.03$) and heavy Dropout ($p=0.50$), suppressing memorization and aligning training and validation loss curves.


In [ ]:
class MLP(nn.Module):
    """MLP for tabular membership inference demo."""
    
    def __init__(self, input_dim: int, large: bool = False, dropout_rate: float = 0.0):
        super(MLP, self).__init__()
        if large:
            # Bigger model to encourage overfitting on the small training subset
            self.net = nn.Sequential(
                nn.Linear(input_dim, 512),
                nn.ReLU(),
                nn.Linear(512, 512),
                nn.ReLU(),
                nn.Linear(512, 256),
                nn.ReLU(),
                nn.Linear(256, 128),
                nn.ReLU(),
                nn.Linear(128, 64),
                nn.ReLU(),
                nn.Linear(64, 2)
            )
        else:
            # Compact model for the regularized baseline
            self.net = nn.Sequential(
                nn.Linear(input_dim, 32),
                nn.ReLU(),
                nn.Dropout(dropout_rate),
                nn.Linear(32, 2)
            )

    def forward(self, x):
        return self.net(x)

In [ ]:
def train_model(model: nn.Module, train_loader: DataLoader, val_loader: DataLoader = None, 
                epochs: int = 10, regularization: str = 'none') -> Tuple[List[float], List[float]]:
    """Train model with optional regularization."""
    model.to(device)
    
    if regularization == 'l2':
        # Use stronger weight decay on the regularized baseline
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9, weight_decay=0.03)
    else:
        optimizer = optim.SGD(model.parameters(), lr=0.01, momentum=0.9)
    
    criterion = nn.CrossEntropyLoss()
    train_losses = []
    val_losses = []
    
    with tqdm(range(epochs), desc="Training", unit="epoch") as pbar:
        for epoch in pbar:
            model.train()
            epoch_loss = 0.0
            for data, target in train_loader:
                data, target = data.to(device), target.to(device)
                optimizer.zero_grad()
                output = model(data)
                loss = criterion(output, target)
                loss.backward()
                optimizer.step()
                epoch_loss += loss.item()
            
            avg_train_loss = epoch_loss / len(train_loader)
            train_losses.append(avg_train_loss)
            
            # Validation
            if val_loader is not None:
                model.eval()
                v_loss = 0.0
                with torch.no_grad():
                    for data, target in val_loader:
                        data, target = data.to(device), target.to(device)
                        output = model(data)
                        v_loss += criterion(output, target).item()
                avg_val_loss = v_loss / len(val_loader)
                val_losses.append(avg_val_loss)
                
                pbar.set_postfix(train_loss=f"{avg_train_loss:.4f}", val_loss=f"{avg_val_loss:.4f}")
            else:
                pbar.set_postfix(loss=f"{avg_train_loss:.4f}")
    
    return train_losses, val_losses

In [ ]:
print("\nTraining overfitted model (larger MLP, no regularization)...")
overfit_model = MLP(input_dim=input_dim, large=True).to(device)
overfit_losses, _ = train_model(overfit_model, train_loader, epochs=200, regularization='none')
print(f"Final training loss: {overfit_losses[-1]:.4f}")

In [ ]:
print("\nTraining regularized model (L2 = 0.03, Dropout = 0.5)...")
reg_model = MLP(input_dim=input_dim, large=False, dropout_rate=0.5).to(device)
reg_losses, val_losses = train_model(reg_model, train_loader, val_loader=val_loader, epochs=20, regularization='l2')
print(f"Final training loss: {reg_losses[-1]:.4f}")

In [ ]:
def evaluate(model: nn.Module, loader: DataLoader) -> float:
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            _, pred = torch.max(output.data, 1)
            correct += (pred == target).sum().item()
            total += target.size(0)
    return 100.0 * correct / total

In [ ]:
overfit_train_acc = evaluate(overfit_model, train_loader)
overfit_test_acc = evaluate(overfit_model, test_loader)

reg_train_acc = evaluate(reg_model, train_loader)
reg_test_acc = evaluate(reg_model, test_loader)

print(f"Model Performance:")
print(f"\nOverfitted Model (large, no reg):")
print(f"  Train accuracy: {overfit_train_acc:.2f}%")
print(f"  Test accuracy: {overfit_test_acc:.2f}%")
print(f"  Overfitting gap: {overfit_train_acc - overfit_test_acc:.2f}%")

print(f"\nRegularized Model (strong L2 + Dropout):")
print(f"  Train accuracy: {reg_train_acc:.2f}%")
print(f"  Test accuracy: {reg_test_acc:.2f}%")
print(f"  Overfitting gap: {reg_train_acc - reg_test_acc:.2f}%")

## **5. Manual Confidence-Based Membership Inference** <a name="confidence"></a>

In black-box settings where the model exposes prediction probabilities, an adversary can exploit two confidence signals:
1. **Max-Softmax Confidence:** $\max_c f_c(\mathbf{x})$
2. **True-Class Confidence:** $f_{y}(\mathbf{x})$ (when the candidate sample's true label $y$ is suspected)

We extract both scores and construct Receiver Operating Characteristic (ROC) curves to quantify the attack advantage:
$$\text{Membership Advantage} = 2 \times (\text{ROC-AUC} - 0.50)$$


In [ ]:
def get_confidence_scores(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Get max softmax confidence for all samples."""
    model.eval()
    confidences = []
    
    with torch.no_grad():
        for data, _ in loader:
            data = data.to(device)
            output = model(data)
            probs = torch.softmax(output, dim=1)
            conf = probs.max(dim=1)[0].cpu().numpy()
            confidences.extend(conf)
    
    return np.array(confidences)


def get_true_class_confidence(model: nn.Module, loader: DataLoader) -> np.ndarray:
    """Get probability assigned to the ground-truth class for each sample."""
    model.eval()
    true_confidences = []
    
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            probs = torch.softmax(model(data), dim=1)
            true_conf = probs[torch.arange(len(target), device=device), target].cpu().numpy()
            true_confidences.extend(true_conf)
    
    return np.array(true_confidences)

In [ ]:
from sklearn.metrics import roc_curve, roc_auc_score

# Compute confidence scores
overfit_train_conf = get_confidence_scores(overfit_model, train_loader)
reg_train_conf = get_confidence_scores(reg_model, train_loader)
overfit_test_conf = get_confidence_scores(overfit_model, test_loader)
reg_test_conf = get_confidence_scores(reg_model, test_loader)

overfit_train_true_conf = get_true_class_confidence(overfit_model, train_loader)
reg_train_true_conf = get_true_class_confidence(reg_model, train_loader)
overfit_test_true_conf = get_true_class_confidence(overfit_model, test_loader)
reg_test_true_conf = get_true_class_confidence(reg_model, test_loader)

# Prepare binary membership ground truth: 1 for training members, 0 for test non-members
y_eval_overfit = np.concatenate([np.ones_like(overfit_train_true_conf), np.zeros_like(overfit_test_true_conf)])
scores_overfit = np.concatenate([overfit_train_true_conf, overfit_test_true_conf])

y_eval_reg = np.concatenate([np.ones_like(reg_train_true_conf), np.zeros_like(reg_test_true_conf)])
scores_reg = np.concatenate([reg_train_true_conf, reg_test_true_conf])

fpr_overfit, tpr_overfit, _ = roc_curve(y_eval_overfit, scores_overfit)
auc_overfit = roc_auc_score(y_eval_overfit, scores_overfit)

fpr_reg, tpr_reg, _ = roc_curve(y_eval_reg, scores_reg)
auc_reg = roc_auc_score(y_eval_reg, scores_reg)

print("=" * 65)
print(f"Manual MIA Confidence Gap & ROC-AUC Summary:")
print("=" * 65)
print(f"  Overfitted Model:  True-Class Gap = {overfit_train_true_conf.mean() - overfit_test_true_conf.mean():.4f} | ROC-AUC = {auc_overfit:.3f} | Advantage = {2*(auc_overfit-0.5):.3f}")
print(f"  Regularized Model: True-Class Gap = {reg_train_true_conf.mean() - reg_test_true_conf.mean():.4f} | ROC-AUC = {auc_reg:.3f} | Advantage = {2*(auc_reg-0.5):.3f}")
print("=" * 65)

# Visualizing Confidence Distributions and ROC Curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: Confidence Distributions
axes[0].hist(overfit_train_true_conf, bins=20, alpha=0.6, label='Overfit Train (Members)', color='crimson')
axes[0].hist(overfit_test_true_conf, bins=20, alpha=0.6, label='Overfit Test (Non-Members)', color='darkred')
axes[0].hist(reg_train_true_conf, bins=20, alpha=0.4, label='Reg Train (Members)', color='steelblue')
axes[0].hist(reg_test_true_conf, bins=20, alpha=0.4, label='Reg Test (Non-Members)', color='navy')
axes[0].set_xlabel('True-Class Confidence Score', fontsize=11)
axes[0].set_ylabel('Sample Frequency', fontsize=11)
axes[0].set_title('Confidence Score Distributions', fontsize=12, fontweight='bold')
axes[0].legend(loc='upper left', fontsize=9)
axes[0].grid(True, linestyle='--', alpha=0.5)

# Panel 2: ROC Curves
axes[1].plot(fpr_overfit, tpr_overfit, color='crimson', lw=2.5, label=f'Overfitted Model (AUC = {auc_overfit:.3f})')
axes[1].plot(fpr_reg, tpr_reg, color='steelblue', lw=2.5, label=f'Regularized Model (AUC = {auc_reg:.3f})')
axes[1].plot([0, 1], [0, 1], 'k--', lw=1.5, label='Random Guessing (AUC = 0.500)')
axes[1].set_xlabel('False Positive Rate (FPR)', fontsize=11)
axes[1].set_ylabel('True Positive Rate (TPR)', fontsize=11)
axes[1].set_title('Membership Inference ROC Curves', fontsize=12, fontweight='bold')
axes[1].legend(loc='lower right', fontsize=10)
axes[1].grid(True, linestyle='--', alpha=0.5)

plt.tight_layout()
plt.show()


## **6. ART Membership Inference Framework** <a name="art"></a>

The [Adversarial Robustness Toolbox (ART)](https://adversarial-robustness-toolbox.readthedocs.io/) provides standardized membership inference implementations.

We encapsulate our PyTorch networks inside `art.estimators.classification.PyTorchClassifier` wrappers and execute `MembershipInferenceBlackBoxRuleBased`, which establishes optimal decision thresholds across model predictions.


In [ ]:
from art.estimators.classification import PyTorchClassifier
from art.attacks.inference.membership_inference import (
    MembershipInferenceBlackBoxRuleBased,
    MembershipInferenceBlackBox,
)

# Wrap both models as ART classifiers
art_overfit = PyTorchClassifier(
    model=overfit_model,
    loss=nn.CrossEntropyLoss(),
    input_shape=(input_dim,),
    nb_classes=2,
    clip_values=(-5.0, 5.0),
)

art_reg = PyTorchClassifier(
    model=reg_model,
    loss=nn.CrossEntropyLoss(),
    input_shape=(input_dim,),
    nb_classes=2,
    clip_values=(-5.0, 5.0),
)
print(f"ART classifiers ready")

# Prepare numpy arrays for ART
X_train_np = X_train.astype(np.float32)  # Already numpy; ensure float32 for ART
y_train_np = y_train
X_test_np = X_test.astype(np.float32)
y_test_np = y_test

# Monkey-patch ART's predict to always use float32 (avoid numpy upcasting issues in HopSkipJump)
_orig_predict_overfit = art_overfit.predict
_orig_predict_reg = art_reg.predict
art_overfit.predict = lambda x, **kw: _orig_predict_overfit(np.asarray(x, dtype=np.float32), **kw)
art_reg.predict = lambda x, **kw: _orig_predict_reg(np.asarray(x, dtype=np.float32), **kw)

In [ ]:
# Rule-based attack: thresholds on prediction confidence
rule_attack_overfit = MembershipInferenceBlackBoxRuleBased(classifier=art_overfit)
rule_attack_reg = MembershipInferenceBlackBoxRuleBased(classifier=art_reg)

# Combine members (train) and non-members (test) for evaluation
X_all = np.vstack([X_train_np, X_test_np])
y_all = np.hstack([y_train_np, y_test_np])
true_membership = np.hstack([np.ones(len(X_train_np)), np.zeros(len(X_test_np))])

# Infer membership
overfit_preds = rule_attack_overfit.infer(X_all, y_all)
reg_preds = rule_attack_reg.infer(X_all, y_all)

# Metrics
from sklearn.metrics import balanced_accuracy_score, roc_auc_score

overfit_bal_acc = balanced_accuracy_score(true_membership, overfit_preds)
reg_bal_acc = balanced_accuracy_score(true_membership, reg_preds)

print("Rule-Based Attack Results (ART):")
print(f"  {'Metric':<30} {'Overfitted':<15} {'Regularized':<15}")
print(f"  {'-'*60}")
print(f"  {'Balanced Accuracy':<30} {overfit_bal_acc*100:<15.1f} {reg_bal_acc*100:<15.1f}")
print(f"  {'Members correct':<30} {overfit_preds[:len(X_train_np)].mean()*100:<15.1f} {reg_preds[:len(X_train_np)].mean()*100:<15.1f}")
print(f"  {'Non-members correct':<30} {(1-overfit_preds[len(X_train_np):]).mean()*100:<15.1f} "
      f"{(1-reg_preds[len(X_train_np):]).mean()*100:<15.1f}")

print(f"\nThe overfitted model identifies {overfit_preds[:len(X_train_np)].mean()*100:.0f}% of members "
      f"(vs {reg_preds[:len(X_train_np)].mean()*100:.0f}% for regularized), "
      f"confirming that overfitting increases membership leakage.")

## **7. Label-Only Decision Boundary Attacks** <a name="label-only"></a>

What if the target API employs **confidence masking** and only returns discrete hard labels (e.g., class `0` vs class `1`) without softmax probabilities?

The **Label-Only Membership Inference Attack** ([Choquette-Choo et al., 2021](https://arxiv.org/abs/2007.14321)) bypasses probability suppression by measuring **adversarial distance to the decision boundary**:
- **Members (training records):** Lie deeper within their respective class decision regions, requiring a larger perturbation $\|\delta\|$ to cross the boundary and flip the predicted label.
- **Non-Members (unseen test records):** Lie closer to decision boundaries on average, requiring smaller perturbations.

Using a small calibration set, ART estimates a critical distance threshold $d^*$ to infer membership from hard decisions alone.


In [ ]:
from art.attacks.inference.membership_inference import LabelOnlyDecisionBoundary
from sklearn.metrics import balanced_accuracy_score

# Split data for calibration (known membership) vs inference (unknown)
n_members = len(X_train_np)
n_nonmembers = len(X_test_np)
cal_members = n_members // 2  # 20
cal_nonmembers = min(n_nonmembers // 2, 50)  # ~43

X_cal = np.vstack([X_train_np[:cal_members], X_test_np[:cal_nonmembers]]).astype(np.float32)
y_cal = np.hstack([y_train_np[:cal_members], y_test_np[:cal_nonmembers]])
member_cal = np.array([1]*cal_members + [0]*cal_nonmembers)

X_eval = np.vstack([X_train_np[cal_members:], X_test_np[cal_nonmembers:]]).astype(np.float32)
y_eval = np.hstack([y_train_np[cal_members:], y_test_np[cal_nonmembers:]])
member_eval = np.array([1]*(n_members - cal_members) + [0]*(n_nonmembers - cal_nonmembers))

print(f"Calibration set: {cal_members} members + {cal_nonmembers} non-members ({len(X_cal)} total)")
print(f"Evaluation set: {n_members - cal_members} members + {n_nonmembers - cal_nonmembers} non-members ({len(X_eval)} total)")

# --- Label-Only attack on overfitted model ---
print("\nCalibrating label-only attack on overfitted model...")
label_only_overfit = LabelOnlyDecisionBoundary(art_overfit)
label_only_overfit.calibrate_distance_threshold(
    X_cal[member_cal == 1], y_cal[member_cal == 1],
    X_cal[member_cal == 0], y_cal[member_cal == 0]
)
pred_overfit = label_only_overfit.infer(X_eval, y_eval)
bal_overfit = balanced_accuracy_score(member_eval, pred_overfit)

# --- Label-Only attack on regularized model ---
print("Calibrating label-only attack on regularized model...")
label_only_reg = LabelOnlyDecisionBoundary(art_reg)
label_only_reg.calibrate_distance_threshold(
    X_cal[member_cal == 1], y_cal[member_cal == 1],
    X_cal[member_cal == 0], y_cal[member_cal == 0]
)
pred_reg = label_only_reg.infer(X_eval, y_eval)
bal_reg = balanced_accuracy_score(member_eval, pred_reg)

# --- Results ---
print(f"\n{'='*60}")
print(f"Label-Only MIA Results (using only hard labels):")
print(f"{'='*60}")
print(f"  {'Model':<30} {'Balanced Accuracy':<20}")
print(f"  {'-'*50}")
print(f"  {'Overfitted':<30} {bal_overfit*100:<20.1f}")
print(f"  {'Regularized':<30} {bal_reg*100:<20.1f}")

# Members correct
n_eval_members = (n_members - cal_members)
overfit_mem_rate = pred_overfit[:n_eval_members].mean() * 100
reg_mem_rate = pred_reg[:n_eval_members].mean() * 100
print(f"\n  Members correctly identified:")
print(f"    Overfitted:  {overfit_mem_rate:.0f}%  (vs {bal_overfit*100:.0f}% balanced)")
print(f"    Regularized: {reg_mem_rate:.0f}%  (vs {bal_reg*100:.0f}% balanced)")

print(f"\n{'='*60}")
print(f"Key insight: Unlike confidence-based attacks, label-only MIA")
print(f"requires no probability outputs \u2014 only the predicted class.")
print(f"This makes it applicable even when APIs only return labels.")
print(f"{'='*60}")

## **8. Hands-on Exploratory Exercises** <a name="exercises"></a>

### **Exercise 1: Model Capacity vs. Privacy Leakage**
Expand the MLP architecture by adding 2 additional hidden layers of width 256. Retrain without regularization and measure the new ROC-AUC for membership inference. Does higher model capacity directly translate to elevated privacy vulnerability?

### **Exercise 2: Training Set Size as a Natural Defense**
Train the overfitted architecture using increasing fractions of the training set ($25\%$, $50\%$, $100\%$). Plot the membership inference AUC as a function of training set size. Explain why more data inherently attenuates individual sample memorization.

### **Exercise 3: Comparative Defense Analysis**
Implement and evaluate four distinct regularization techniques on the overfitted architecture:
1. $L_1$ weight regularization (sparsity)
2. $L_2$ weight decay (shrinkage)
3. Dropout ($p=0.5$)
4. Label Smoothing (soft cross-entropy targets)

Which technique produces the lowest membership advantage while maintaining acceptable test accuracy?

### **Exercise 4: Label-Only vs. Confidence-Based Attack Comparison**
Create a summary table comparing the empirical Balanced Accuracy of:
- Manual True-Class Confidence MIA
- ART Rule-Based Black-Box MIA
- ART Label-Only Decision Boundary MIA

Under what operational conditions would an adversary choose each attack method?

### **Exercise 5: Calibration Sample Sensitivity**
Vary the number of calibration samples available to the `LabelOnlyDecisionBoundary` attack ($5, 10, 25, 50$). How does calibration sample scarcity impact the attack's balanced accuracy?


## **9. Conclusion & Next Steps** <a name="conclusion"></a>

### **Core Takeaways**
1. **Generalization Gap $\equiv$ Privacy Vulnerability:** Membership inference attacks fundamentally exploit empirical generalization error ($Acc_{train} - Acc_{test}$). Models that memorize training data leak significant confidence and loss disparities.
2. **Confidence Thresholding is Highly Effective:** When softmax probabilities are exposed, thresholding on true-class confidence achieves high attack accuracy on overfitted models ($AUC > 0.85$).
3. **Hard-Label Defenses Are Insufficient:** Suppressing probabilities to output only discrete class labels forces attackers to pivot to label-only decision boundary attacks (e.g. Choquette-Choo et al., 2021), which probe distance-to-boundary via query perturbations.
4. **Regularization Helps, But DP is Provable:** Classical techniques (weight decay, dropout, label smoothing) reduce empirical privacy leakage by penalizing overfitting, but do not provide formal guarantees. Provable membership privacy requires Differential Privacy (DP-SGD), covered in Module 7.

### **MITRE ATLAS Mapping**
- **Tactic:** [AML.TA0005: Exfiltration](https://atlas.mitre.org/tactics/AML.TA0005)
- **Technique:** [AML.T0024: Infer Training Data Membership](https://atlas.mitre.org/techniques/AML.T0024)

### **Key References**
- Shokri, R., Stronati, M., Song, C., & Shmatikov, V. (2017). *Membership Inference Attacks Against Machine Learning Models*. IEEE S&P. [arXiv:1610.05820](https://arxiv.org/abs/1610.05820).
- Yeom, S., Giacomelli, I., Fredrikson, M., & Jha, S. (2018). *Privacy Risk in Machine Learning: Analyzing the Connection to Overfitting*. IEEE CSF. [arXiv:1709.01604](https://arxiv.org/abs/1709.01604).
- Choquette-Choo, C. A., Tramer, F., Carlini, N., & Papernot, N. (2021). *Label-Only Membership Inference Attacks*. ICML. [arXiv:2007.14321](https://arxiv.org/abs/2007.14321).
